In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Download stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# ============================================
# STEP 1: LOAD AND COMBINE ALL DATASETS
# ============================================

print("="*60)
print("LOADING AND COMBINING DATASETS")
print("="*60)

datasets = []

# 1. Load fake.csv
try:
    fake = pd.read_csv("../datasets/fake.csv")
    fake['label'] = 0
    fake['source'] = 'fake'
    # Ensure text column exists
    if 'text' not in fake.columns:
        fake['text'] = fake['title'] + ' ' + fake.get('description', '')
    datasets.append(fake)
    print(f"✓ Loaded fake.csv: {fake.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load fake.csv: {e}")

# 2. Load True.csv
try:
    true = pd.read_csv("../datasets/True.csv")
    true['label'] = 1
    true['source'] = 'true'
    if 'text' not in true.columns:
        true['text'] = true['title'] + ' ' + true.get('description', '')
    datasets.append(true)
    print(f"✓ Loaded True.csv: {true.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load True.csv: {e}")

# 3. Load AG_Train.csv
try:
    ag_train = pd.read_csv("../datasets/AG_Train.csv")
    print(f"AG_Train columns: {ag_train.columns.tolist()}")
    
    # Map labels
    if 'Class Index' in ag_train.columns:
        ag_train['label'] = ag_train['Class Index'].apply(lambda x: 1 if x in [3, 4] else 0)
    elif 'class' in ag_train.columns:
        ag_train['label'] = ag_train['class'].apply(lambda x: 1 if x in [3, 4] else 0)
    elif 'label' in ag_train.columns:
        ag_train['label'] = ag_train['label'].apply(lambda x: 1 if x in [3, 4] else 0)
    else:
        first_col = ag_train.columns[0]
        ag_train['label'] = ag_train[first_col].apply(lambda x: 1 if x in [3, 4] else 0)
    
    # Combine Title and Description into one text column
    # AG dataset has: Class Index, Title, Description
    if 'Title' in ag_train.columns and 'Description' in ag_train.columns:
        ag_train['text'] = ag_train['Title'].fillna('') + ' ' + ag_train['Description'].fillna('')
    elif 'title' in ag_train.columns and 'description' in ag_train.columns:
        ag_train['text'] = ag_train['title'].fillna('') + ' ' + ag_train['description'].fillna('')
    elif len(ag_train.columns) >= 3:
        # Assume columns are: class, title, description
        col_names = ag_train.columns.tolist()
        ag_train['text'] = ag_train[col_names[1]].fillna('') + ' ' + ag_train[col_names[2]].fillna('')
    else:
        # If only one text column
        text_col = ag_train.columns[1] if len(ag_train.columns) > 1 else ag_train.columns[0]
        ag_train['text'] = ag_train[text_col]
    
    ag_train['source'] = 'AG_Train'
    datasets.append(ag_train)
    print(f"✓ Loaded AG_Train.csv: {ag_train.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load AG_Train.csv: {e}")

# 4. Load AG_Test.csv
try:
    ag_test = pd.read_csv("../datasets/AG_Test.csv")
    print(f"AG_Test columns: {ag_test.columns.tolist()}")
    
    if 'Class Index' in ag_test.columns:
        ag_test['label'] = ag_test['Class Index'].apply(lambda x: 1 if x in [3, 4] else 0)
    elif 'class' in ag_test.columns:
        ag_test['label'] = ag_test['class'].apply(lambda x: 1 if x in [3, 4] else 0)
    elif 'label' in ag_test.columns:
        ag_test['label'] = ag_test['label'].apply(lambda x: 1 if x in [3, 4] else 0)
    else:
        first_col = ag_test.columns[0]
        ag_test['label'] = ag_test[first_col].apply(lambda x: 1 if x in [3, 4] else 0)
    
    # Combine Title and Description into one text column
    if 'Title' in ag_test.columns and 'Description' in ag_test.columns:
        ag_test['text'] = ag_test['Title'].fillna('') + ' ' + ag_test['Description'].fillna('')
    elif 'title' in ag_test.columns and 'description' in ag_test.columns:
        ag_test['text'] = ag_test['title'].fillna('') + ' ' + ag_test['description'].fillna('')
    elif len(ag_test.columns) >= 3:
        col_names = ag_test.columns.tolist()
        ag_test['text'] = ag_test[col_names[1]].fillna('') + ' ' + ag_test[col_names[2]].fillna('')
    else:
        text_col = ag_test.columns[1] if len(ag_test.columns) > 1 else ag_test.columns[0]
        ag_test['text'] = ag_test[text_col]
    
    ag_test['source'] = 'AG_Test'
    datasets.append(ag_test)
    print(f"✓ Loaded AG_Test.csv: {ag_test.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load AG_Test.csv: {e}")

# Combine all datasets
df = pd.concat(datasets, ignore_index=True)

print("-"*60)
print(f"Total combined rows: {len(df):,}")

# Check if text column exists
print(f"\nColumns in final dataset: {df.columns.tolist()}")
print(f"Sample text from first row: {df['text'].iloc[0][:100]}...")

LOADING AND COMBINING DATASETS
✓ Loaded fake.csv: 23,481 rows
✓ Loaded True.csv: 21,417 rows
AG_Train columns: ['Class Index', 'Title', 'Description']
✓ Loaded AG_Train.csv: 120,000 rows
AG_Test columns: ['Class Index', 'Title', 'Description']
✓ Loaded AG_Test.csv: 7,600 rows
------------------------------------------------------------
Total combined rows: 172,498

Columns in final dataset: ['title', 'text', 'subject', 'date', 'label', 'source', 'Class Index', 'Title', 'Description']
Sample text from first row: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had...


In [5]:
# ============================================
# STEP 2: DATA EXPLORATION
# ============================================

print("\n" + "="*60)
print("DATA EXPLORATION")
print("="*60)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

print("\nLabel distribution:")
label_counts = df['label'].value_counts()
for label, count in label_counts.items():
    label_name = "True" if label == 1 else "Fake"
    print(f"  {label_name}: {count:,} ({count/len(df)*100:.2f}%)")

print("\nSource distribution:")
print(df['source'].value_counts())

print(fake.shape)
print(true.shape)


DATA EXPLORATION
Dataset shape: (172498, 9)
Columns: ['title', 'text', 'subject', 'date', 'label', 'source', 'Class Index', 'Title', 'Description']

Label distribution:
  Fake: 87,281 (50.60%)
  True: 85,217 (49.40%)

Source distribution:
source
AG_Train    120000
fake         23481
true         21417
AG_Test       7600
Name: count, dtype: int64
(23481, 6)
(21417, 6)


In [6]:
# ============================================
# STEP 3: TEXT CLEANING
# ============================================

stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [7]:
# ============================================
# STEP 4: CLEAN THE TEXT DATA
# ============================================

print("\n" + "="*60)
print("CLEANING TEXT DATA")
print("="*60)

df["clean_text"] = df["text"].apply(clean_text)

before = len(df)
df = df[df['clean_text'].str.len() > 0]
after = len(df)
print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning: {after:,}")
print(f"Rows removed: {before - after:,}")

before = len(df)
df = df.drop_duplicates(subset=['clean_text'])
after = len(df)
print(f"\nRows before removing duplicates: {before:,}")
print(f"Rows after removing duplicates: {after:,}")
print(f"Duplicates removed: {before - after:,}")

df[["text","clean_text"]].head()


CLEANING TEXT DATA
Rows before cleaning: 172,498
Rows after cleaning: 171,782
Rows removed: 716

Rows before removing duplicates: 171,782
Rows after removing duplicates: 165,707
Duplicates removed: 6,075


,text,clean_text
0,Donald Trump just couldn t wish all Americans ...,donald trump wish americans happy new year lea...
1,House Intelligence Committee Chairman Devin Nu...,house intelligence committee chairman devin nu...
2,"On Friday, it was revealed that former Milwauk...",friday revealed former milwaukee sheriff david...
3,"On Christmas day, Donald Trump announced that ...",christmas day donald trump announced would bac...
4,Pope Francis used his annual Christmas Day mes...,pope francis used annual christmas day message...


In [8]:
# ============================================
# STEP 5: TRAIN-TEST SPLIT
# ============================================

print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size: {len(X_train):,}")
print(f"Testing set size: {len(X_test):,}")
print(f"Train/Test split: {len(X_train)/len(X)*100:.1f}% / {len(X_test)/len(X)*100:.1f}%")

print("\nTraining set label distribution:")
for label, count in y_train.value_counts().items():
    label_name = "True" if label == 1 else "Fake"
    print(f"  {label_name}: {count:,} ({count/len(y_train)*100:.2f}%)")

print("\nTesting set label distribution:")
for label, count in y_test.value_counts().items():
    label_name = "True" if label == 1 else "Fake"
    print(f"  {label_name}: {count:,} ({count/len(y_test)*100:.2f}%)")


TRAIN-TEST SPLIT
Training set size: 132,565
Testing set size: 33,142
Train/Test split: 80.0% / 20.0%

Training set label distribution:
  True: 67,737 (51.10%)
  Fake: 64,828 (48.90%)

Testing set label distribution:
  True: 16,935 (51.10%)
  Fake: 16,207 (48.90%)


In [10]:
# ============================================
# STEP 6: TF-IDF VECTORIZATION
# ============================================

print("\n" + "="*60)
print("TF-IDF VECTORIZATION")
print("="*60)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.9
)

print("Fitting vectorizer on training data...")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training data shape: {X_train_tfidf.shape}")
print(f"Testing data shape: {X_test_tfidf.shape}")
print(f"Number of features: {len(vectorizer.get_feature_names_out()):,}")


TF-IDF VECTORIZATION
Fitting vectorizer on training data...
Training data shape: (132565, 10000)
Testing data shape: (33142, 10000)
Number of features: 10,000


In [11]:
# ============================================
# STEP 7: TRAIN MULTIPLE MODELS
# ============================================

print("\n" + "="*60)
print("TRAINING MODELS")
print("="*60)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(max_iter=1000, random_state=42, dual=False)
}

best_accuracy = 0
best_model = None
best_model_name = ""
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    try:
        model.fit(X_train_tfidf, y_train)
        predictions = model.predict(X_test_tfidf)
        accuracy = accuracy_score(y_test, predictions)
        results[name] = accuracy
        print(f"  ✓ {name} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model
            best_model_name = name
    except Exception as e:
        print(f"  ✗ {name} failed: {e}")

print("\n" + "-"*60)
print("MODEL COMPARISON RESULTS:")
print("-"*60)
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
for rank, (name, acc) in enumerate(sorted_results, 1):
    print(f"  #{rank}. {name}: {acc:.4f} ({acc*100:.2f}%)")

print(f"\n✓ Best Model: {best_model_name} with {best_accuracy*100:.2f}% accuracy")
model = best_model


TRAINING MODELS

Training Logistic Regression...
  ✓ Logistic Regression Accuracy: 0.9472 (94.72%)

Training Random Forest...
  ✓ Random Forest Accuracy: 0.9532 (95.32%)

Training Naive Bayes...
  ✓ Naive Bayes Accuracy: 0.9153 (91.53%)

Training Linear SVM...
  ✓ Linear SVM Accuracy: 0.9475 (94.75%)

------------------------------------------------------------
MODEL COMPARISON RESULTS:
------------------------------------------------------------
  #1. Random Forest: 0.9532 (95.32%)
  #2. Linear SVM: 0.9475 (94.75%)
  #3. Logistic Regression: 0.9472 (94.72%)
  #4. Naive Bayes: 0.9153 (91.53%)

✓ Best Model: Random Forest with 95.32% accuracy


In [12]:
# ============================================
# STEP 8: EVALUATE BEST MODEL
# ============================================

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['Fake', 'True']))

cm = confusion_matrix(y_test, predictions)
cm_df = pd.DataFrame(
    cm, 
    index=['Actual Fake', 'Actual True'], 
    columns=['Predicted Fake', 'Predicted True']
)
print("\nConfusion Matrix:")
print(cm_df)

tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\nAdditional Metrics:")
print(f"  Precision (True class): {precision:.4f}")
print(f"  Recall (True class): {recall:.4f}")
print(f"  F1 Score (True class): {f1:.4f}")
print(f"  False Positive Rate: {fp / (fp + tn):.4f}")
print(f"  False Negative Rate: {fn / (fn + tp):.4f}")


MODEL EVALUATION
Accuracy: 0.9532 (95.32%)

Classification Report:
              precision    recall  f1-score   support

        Fake       0.95      0.95      0.95     16207
        True       0.96      0.95      0.95     16935

    accuracy                           0.95     33142
   macro avg       0.95      0.95      0.95     33142
weighted avg       0.95      0.95      0.95     33142


Confusion Matrix:
             Predicted Fake  Predicted True
Actual Fake           15452             755
Actual True             796           16139

Additional Metrics:
  Precision (True class): 0.9553
  Recall (True class): 0.9530
  F1 Score (True class): 0.9542
  False Positive Rate: 0.0466
  False Negative Rate: 0.0470


In [13]:
# ============================================
# STEP 9: SAVE MODEL AND VECTORIZER
# ============================================

print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

with open("fake_news_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("✓ Model saved to: fake_news_model.pkl")

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("✓ Vectorizer saved to: tfidf_vectorizer.pkl")


SAVING MODEL
✓ Model saved to: fake_news_model.pkl
✓ Vectorizer saved to: tfidf_vectorizer.pkl


In [21]:
# ============================================
# STEP 10: TEST PREDICTIONS
# ============================================

print("\n" + "="*60)
print("TESTING PREDICTIONS")
print("="*60)

def predict_news(news_text):
    if isinstance(news_text, str):
        news_text = [news_text]
    cleaned_text = [clean_text(text) for text in news_text]
    vectors = vectorizer.transform(cleaned_text)
    predictions = model.predict(vectors)
    probabilities = model.predict_proba(vectors)
    return predictions, probabilities

def predict_with_confidence(news_text):
    pred, prob = predict_news(news_text)
    results = []
    for i, (prediction, probs) in enumerate(zip(pred, prob)):
        confidence = max(probs)
        label = "True" if prediction == 1 else "Fake"
        result = {
            'text': news_text[i] if isinstance(news_text, list) else news_text,
            'prediction': prediction,
            'label': label,
            'confidence': confidence,
            'fake_probability': probs[0],
            'true_probability': probs[1]
        }
        results.append(result)
    return results[0] if len(results) == 1 else results

test_articles = [
    "Aliens attacked Lahore and kidnapped the Prime Minister",
    "Imran khan is alive and well",
    "The US Federal Reserve announced interest rate cuts today",
    "Scientists discover cure for cancer in clinical trials"
]

print("\nSample Predictions:")
print("-"*60)
for article in test_articles:
    result = predict_with_confidence(article)
    print(f"\nArticle: {article[:80]}...")
    print(f"Prediction: {result['label']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Fake Probability: {result['fake_probability']:.2%}")
    print(f"True Probability: {result['true_probability']:.2%}")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)


TESTING PREDICTIONS

Sample Predictions:
------------------------------------------------------------

Article: Aliens attacked Lahore and kidnapped the Prime Minister...
Prediction: Fake
Confidence: 100.00%
Fake Probability: 100.00%
True Probability: 0.00%

Article: Imran khan is alive and well...
Prediction: Fake
Confidence: 89.84%
Fake Probability: 89.84%
True Probability: 10.16%

Article: The US Federal Reserve announced interest rate cuts today...
Prediction: True
Confidence: 96.00%
Fake Probability: 4.00%
True Probability: 96.00%

Article: Scientists discover cure for cancer in clinical trials...
Prediction: True
Confidence: 77.00%
Fake Probability: 23.00%
True Probability: 77.00%

TRAINING COMPLETE!


In [ ]:
# ============================================
# QUICK PREDICTION FUNCTION - USE THIS LATER
# ============================================

def quick_predict(news_text):
    """Quick function to predict if news is fake or true"""
    cleaned = clean_text(news_text)
    vector = vectorizer.transform([cleaned])
    pred = model.predict(vector)[0]
    prob = model.predict_proba(vector)[0]
    label = "True" if pred == 1 else "Fake"
    confidence = max(prob)
    return {
        'label': label,
        'confidence': confidence,
        'fake_probability': prob[0],
        'true_probability': prob[1]
    }

# Test it
print(quick_predict("Breaking: Secret alien base discovered on the Moon!"))